# Incremental demo — data setup (volume + 2 CSV batches)

Feeds the **incremental pipeline** version of the reviews demo. Instead of inline `VALUES` data, the reviews arrive as **files in a Unity Catalog volume** — the way real batch data lands:

* `batch_1_reviews.csv` — the original **25 reviews** (identical to notebook 02's bronze load)
* `batch_2_reviews.csv` — **15 new reviews** (ids 26–40)

**The demo flow:**
1. Before the session, run this notebook: batch 1 lands in `raw_reviews` (the folder the pipeline watches); batch 2 waits in `staging_reviews` — **outside** the watched folder, so the first refresh cannot see it yet
2. Pipeline **refresh 1**: Auto Loader ingests batch 1 only — 25 rows flow bronze → silver → gold
3. Live, mid-session: run the last cell of this notebook (or drag the file into the volume in Catalog Explorer) — batch 2 **lands**, exactly like a new arrival from a source system
4. Pipeline **refresh 2**: Auto Loader's checkpoint knows batch 1 is already ingested, so it reads **only** the new file — just 15 rows flow through, and the AI functions run on only those 15

**Batch 2 contains the same traps as batch 1**, so the DQ rules and masking have fresh work to do: one junk row (`meh` → `too_short`), one **exact duplicate** of review 8, one **near-duplicate** of review 4, and one PII row with a phone number.

Run this notebook top to bottom **before** the pipeline. It is idempotent — re-running overwrites both files with identical content, so Auto Loader never sees "new" duplicates.

In [0]:
%sql
-- Two volumes make the incremental demo clean:
--   raw_reviews:     the folder the pipeline WATCHES (Auto Loader source)
--   staging_reviews: where batch 2 waits BEFORE it "lands" — it must stay OUTSIDE
--                    the watched folder, or refresh 1 would ingest both files at once
-- A Unity Catalog volume is a governed folder: permissions, lineage and audit
-- apply to the files inside, just like tables.
CREATE SCHEMA IF NOT EXISTS workspace.food_delivery_demo;
CREATE VOLUME IF NOT EXISTS workspace.food_delivery_demo.raw_reviews;
CREATE VOLUME IF NOT EXISTS workspace.food_delivery_demo.staging_reviews;

In [0]:
# Batch 1: the original 25 reviews — identical to notebook 02's bronze load.
# Written as ONE CSV file with a deterministic name (not a Spark part-file folder),
# so re-running this notebook overwrites the same file instead of creating new ones
# that Auto Loader would ingest twice.
import csv

batch_1 = [
    (1, "O1001", "2026-11-01 12:10", "London", "UK", "en", 1, "Food arrived 70 minutes late and completely cold. Very disappointed.", "c01@ex.com", "negative", "delivery_delay"),
    (2, "O1002", "2026-11-01 13:05", "Austin", "US", "en", 5, "Amazing biryani, still hot and the rider was super friendly!", "c02@ex.com", "positive", "none"),
    (3, "O1003", "2026-11-01 19:40", "Toronto", "CA", "en", 2, "Ordered two burgers but only got one. Want a refund please.", "c03@ex.com", "negative", "wrong_or_missing_item"),
    (4, "O1004", "2026-11-02 20:15", "Chicago", "US", "en", 3, "Tastes great but the delivery fee is way too high now.", "c04@ex.com", "negative", "price"),
    (5, "O1005", "2026-11-02 21:00", "Miami", "US", "en", 3, "ok", "c05@ex.com", "neutral", "none"),
    (6, "O1006", "2026-11-03 12:30", "Austin", "US", "en", 2, "The app kept crashing every time I tried to pay.", "c06@ex.com", "negative", "app_issue"),
    (7, "O1007", "2026-11-03 18:45", "Austin", "US", "en", 5, "Amazing biryani, still hot and the rider was super friendly!", "c07@ex.com", "positive", "none"),
    (8, "O1008", "2026-11-03 19:10", "Mumbai", "IN", "hi", 1, "Khana ek ghante late aaya aur bilkul thanda tha.", "c08@ex.com", "negative", "delivery_delay"),
    (9, "O1009", "2026-11-04 13:15", "Delhi", "IN", "hi", 5, "Biryani bahut swadisht thi, rider bhi bahut accha tha!", "c09@ex.com", "positive", "none"),
    (10, "O1010", "2026-11-04 14:00", "Pune", "IN", "hi", 2, "Maine do pizza order kiye the, sirf ek mila.", "c10@ex.com", "negative", "wrong_or_missing_item"),
    (11, "O1011", "2026-11-04 20:30", "Delhi", "IN", "hi", 2, "Khana bilkul kharab tha, taste achha nahi laga.", "c11@ex.com", "negative", "food_quality"),
    (12, "O1012", "2026-11-05 12:20", "Mexico City", "MX", "es", 1, "La comida llegó una hora tarde y fría.", "c12@ex.com", "negative", "delivery_delay"),
    (13, "O1013", "2026-11-05 13:10", "Madrid", "ES", "es", 5, "¡La pizza estaba increíble y llegó caliente!", "c13@ex.com", "positive", "none"),
    (14, "O1014", "2026-11-05 19:55", "Guadalajara", "MX", "es", 2, "Pedí dos tacos y solo llegó uno. Quiero mi dinero.", "c14@ex.com", "negative", "wrong_or_missing_item"),
    (15, "O1015", "2026-11-06 12:05", "Bogota", "CO", "es", 3, "El costo de envío es demasiado alto.", "c15@ex.com", "negative", "price"),
    (16, "O1016", "2026-11-06 13:00", "Berlin", "DE", "de", 1, "Das Essen kam eine Stunde zu spät und kalt an.", "c16@ex.com", "negative", "delivery_delay"),
    (17, "O1017", "2026-11-06 19:30", "Munich", "DE", "de", 5, "Sehr leckeres Essen und schnelle Lieferung!", "c17@ex.com", "positive", "none"),
    (18, "O1018", "2026-11-06 20:10", "Hamburg", "DE", "de", 2, "Ich habe zwei Burger bestellt, aber nur einen bekommen.", "c18@ex.com", "negative", "wrong_or_missing_item"),
    (19, "O1019", "2026-11-07 12:15", "Berlin", "DE", "de", 1, "Rufen Sie mich unter 0176 1234567 an. Das Essen war kalt und hat furchtbar geschmeckt.", "c19@ex.com", "negative", "food_quality"),
    (20, "O1020", "2026-11-07 12:40", "Tokyo", "JP", "ja", 1, "料理が1時間遅れて届き、冷めていました。", "c20@ex.com", "negative", "delivery_delay"),
    (21, "O1021", "2026-11-07 13:25", "Osaka", "JP", "ja", 5, "とても美味しかったです!スムーズな配達でした。", "c21@ex.com", "positive", "none"),
    (22, "O1022", "2026-11-07 18:50", "Tokyo", "JP", "ja", 2, "注文した2つのバーガーのうち1つしか届きませんでした。", "c22@ex.com", "negative", "wrong_or_missing_item"),
    (23, "O1023", "2026-11-07 19:05", "Sao Paulo", "BR", "pt", 1, "A comida chegou uma hora atrasada e fria.", "c23@ex.com", "negative", "delivery_delay"),
    (24, "O1024", "2026-11-07 19:40", "Sao Paulo", "BR", "pt", 5, "Muito saboroso, entrega rápida e pontual!", "c24@ex.com", "positive", "none"),
    (25, "O1025", "2026-11-07 20:00", "Lisbon", "PT", "pt", 1, "Meu nome é Ana Silva, ligue para 21 98765-4321. O pedido nunca chegou.", "c25@ex.com", "negative", "delivery_delay"),
]

header = ["review_id", "order_id", "review_ts", "city", "country", "language", "rating",
          "review_text", "customer_email", "expected_sentiment", "expected_issue"]

path = "/Volumes/workspace/food_delivery_demo/raw_reviews/batch_1_reviews.csv"
with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(batch_1)

print(f"batch_1_reviews.csv written: {len(batch_1)} rows -> {path}")

In [0]:
# Batch 2: 15 NEW reviews (ids 26-40), written to the STAGING volume —
# deliberately OUTSIDE the folder the pipeline watches. It "lands" in raw_reviews
# live during the demo (see the last cell of this notebook).
# Deliberately contains the same traps as batch 1 so the DQ rules have fresh work:
#   id 35: exact duplicate of review 8 (Hindi)  -> duplicate_text rule
#   id 36: near-duplicate of review 4           -> ai_similarity territory
#   id 37: phone number inside the text          -> ai_mask before any model call
#   id 40: 'meh' (3 chars)                       -> too_short rule
import csv

batch_2 = [
    (26, "O1026", "2026-11-08 12:10", "Manchester", "UK", "en", 5, "Fast delivery and the fish and chips were piping hot!", "c26@ex.com", "positive", "none"),
    (27, "O1027", "2026-11-08 13:15", "Jaipur", "IN", "hi", 1, "Khana do ghante late aaya, bilkul thanda aur bekar.", "c27@ex.com", "negative", "delivery_delay"),
    (28, "O1028", "2026-11-08 14:00", "Barcelona", "ES", "es", 2, "La app se bloqueó tres veces al pagar. Frustrante.", "c28@ex.com", "negative", "app_issue"),
    (29, "O1029", "2026-11-08 15:30", "Tokyo", "JP", "ja", 3, "味はまあまあだったが、配達料が高すぎる。", "c29@ex.com", "negative", "price"),
    (30, "O1030", "2026-11-08 16:45", "Berlin", "DE", "de", 5, "Das Essen war ausgezeichnet und kam pünktlich!", "c30@ex.com", "positive", "none"),
    (31, "O1031", "2026-11-08 17:20", "Austin", "US", "en", 1, "The app charged me twice for one order. Terrible service.", "c31@ex.com", "negative", "app_issue"),
    (32, "O1032", "2026-11-08 18:00", "Bogota", "CO", "es", 2, "Pedí una ensalada y llegó una hamburguesa. Quiero mi dinero.", "c32@ex.com", "negative", "wrong_or_missing_item"),
    (33, "O1033", "2026-11-08 19:10", "Osaka", "JP", "ja", 1, "2時間待っても届きませんでした。二度と注文しません。", "c33@ex.com", "negative", "delivery_delay"),
    (34, "O1034", "2026-11-08 20:00", "Lisbon", "PT", "pt", 5, "Excelente comida e entrega super rápida!", "c34@ex.com", "positive", "none"),
    (35, "O1035", "2026-11-08 20:30", "Mumbai", "IN", "hi", 1, "Khana ek ghante late aaya aur bilkul thanda tha.", "c35@ex.com", "negative", "delivery_delay"),
    (36, "O1036", "2026-11-08 21:00", "Chicago", "US", "en", 3, "The delivery fee is way too high now.", "c36@ex.com", "negative", "price"),
    (37, "O1037", "2026-11-08 21:30", "Hamburg", "DE", "de", 1, "Rufen Sie mich unter 0176 9876543 an. Das Essen war kalt und hat furchtbar geschmeckt.", "c37@ex.com", "negative", "food_quality"),
    (38, "O1038", "2026-11-08 22:00", "Sao Paulo", "BR", "pt", 2, "Faltaram dois itens do meu pedido e o suporte não respondeu.", "c38@ex.com", "negative", "wrong_or_missing_item"),
    (39, "O1039", "2026-11-08 22:30", "Madrid", "ES", "es", 5, "¡Mejor app de delivery! Todo perfecto.", "c39@ex.com", "positive", "none"),
    (40, "O1040", "2026-11-08 23:00", "Miami", "US", "en", 3, "meh", "c40@ex.com", "neutral", "none"),
]

path = "/Volumes/workspace/food_delivery_demo/staging_reviews/batch_2_reviews.csv"
with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(batch_2)

print(f"batch_2_reviews.csv written: {len(batch_2)} rows -> {path}")

In [0]:
%sql
-- Verify: batch 1 is waiting IN the watched folder (25 rows),
-- batch 2 is held OUTSIDE it (15 rows) — so refresh 1 can only see batch 1.
SELECT 'raw_reviews (watched)'   AS folder, count(*) AS rows
FROM   read_files('/Volumes/workspace/food_delivery_demo/raw_reviews/', format => 'csv', header => 'true')
UNION ALL
SELECT 'staging_reviews (held)', count(*)
FROM   read_files('/Volumes/workspace/food_delivery_demo/staging_reviews/', format => 'csv', header => 'true');

In [0]:
%sql
-- Preview batch 2 (staged, not yet landed): the trap rows —
-- duplicate (35), near-duplicate (36), PII (37), junk (40).
SELECT review_id, language, rating, review_text, expected_issue
FROM   read_files('/Volumes/workspace/food_delivery_demo/staging_reviews/',
                  format => 'csv', header => 'true')
WHERE  review_id IN (35, 36, 37, 40)
ORDER  BY review_id;

In [0]:
# DEMO MOMENT — run this LIVE, after the pipeline's FIRST refresh has fully
# processed batch 1. It "lands" batch 2 in the watched folder — exactly like a
# new file arriving from a source system. Then start the pipeline's SECOND
# refresh: Auto Loader's checkpoint knows batch_1 was already ingested, so only
# batch_2_reviews.csv is read — just 15 new rows flow through the pipeline,
# and the AI functions run on only those 15 rows.
#
dbutils.fs.cp("/Volumes/workspace/food_delivery_demo/staging_reviews/batch_2_reviews.csv",
               "/Volumes/workspace/food_delivery_demo/raw_reviews/batch_2_reviews.csv")
print("batch 2 landed — start pipeline refresh 2")